In [2]:
import random
import rich
import sys
import json
from collections import Counter
import os
sys.path.append("/home/wenkail/diplomacy/sotopia-diplomacy/src")
sys.path.append("../")
from tqdm import tqdm
from sotopia.database import AgentProfile, EpisodeLog, EnvironmentProfile
import argparse
from tqdm import tqdm
import os
import json
import pandas as pd
import pdb
import random

In [3]:
folder_path = "/data/user_data/wenkail/sotopia_diplomacy/clean_global_whole_games"
file_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)

games = []
num = 0
for file_path in tqdm(file_paths):
    with open(file_path, 'r') as f:
        games.append(json.load(f))

  0%|          | 0/3047 [00:00<?, ?it/s]

100%|██████████| 3047/3047 [00:24<00:00, 125.43it/s]


In [4]:
valid_countries = ['Austria', 'England', 'France', 'Germany', 'Italy', 'Russia', 'Turkey']
upper_countries = [c.upper() for c in valid_countries]

In [5]:
threshold_dict = {}
for threshold in tqdm(range(6, 36)):
    choice_phases_list = []
    for game in games:
        
        for phase in game['phases']:
            # Only consider phases from 1903 to 1907 that end with 'M'
            phase_name = phase['name']
            if not (phase_name.endswith('M') and any(year in phase_name for year in ['1903', '1904', '1905', '1906', '1907'])):
                continue
                
            choice_phase = {}
            counter = Counter()
            messages = phase['messages']
            counter = Counter()

            for msg in messages:
                pair = tuple(sorted([msg['sender'], msg['recipient']]))
                counter[pair] += 1
            choice_phase['game_id'] = games[0]['id']
            choice_phase['phase_name'] = phase['name']
            for pair, count in counter.items():
                if count >= threshold:
                    choice_phase['countries'] = list(pair)
                    choice_phase['message_count'] = count
                    choice_phases_list.append(choice_phase)
    threshold_dict[threshold] = len(choice_phases_list)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:13<00:00,  2.25it/s]


In [6]:
threshold_dict

{6: 31619,
 7: 23748,
 8: 18102,
 9: 14188,
 10: 11353,
 11: 9164,
 12: 7426,
 13: 6091,
 14: 5051,
 15: 4259,
 16: 3601,
 17: 3063,
 18: 2611,
 19: 2241,
 20: 1943,
 21: 1696,
 22: 1471,
 23: 1317,
 24: 1169,
 25: 1049,
 26: 914,
 27: 817,
 28: 724,
 29: 646,
 30: 581,
 31: 525,
 32: 474,
 33: 431,
 34: 391,
 35: 353}

In [7]:
from collections import Counter

threshold_dict = {}
choice_phases_list = []

for game in games:
    for phase in game['phases']:
        # Only consider phases from 1903 to 1907 that end with 'M'
        phase_name = phase['name']
        if not (phase_name.endswith('M') and any(year in phase_name for year in ['1903', '1904', '1905', '1906', '1907'])):
            continue
            
        messages = phase['messages']
        counter = Counter()

        for msg in messages:
            pair = tuple(sorted([msg['sender'], msg['recipient']]))
            counter[pair] += 1

        for pair, count in counter.items():
            if count >= 6 and count <= 36:
                choice_phase = {}
                choice_phase['game_id'] = game['id']
                choice_phase['phase'] = phase['name']
                countries = list(pair)
                countries = [c.capitalize() for c in countries]
                choice_phase['countries'] = countries
                choice_phase['message_count'] = count
                choice_phases_list.append(choice_phase)
                # Remove the `break` to consider all pairs meeting the threshold


In [9]:
choice_phases_list[1]

{'game_id': '26554',
 'phase': 'S1903M',
 'countries': ['Austria', 'Russia'],
 'message_count': 7}

In [10]:
file_path = "/home/wenkail/diplomacy/sotopia-diplomacy/src/environment_profiles_generation/whole_choice_phase_list.json"
with open(file_path, 'w') as f:
    json.dump(choice_phases_list,f,indent=2)